In [1]:
# Ignore  the warnings 
import warnings 
warnings.filterwarnings('always') 
warnings.filterwarnings('ignore') 

In [2]:
# data visualisation and manipulation 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import style
import seaborn as sns
 
#configure
# sets matplotlib to inline and displays graphs below the corressponding cell. 
%matplotlib inline  
style.use('fivethirtyeight')
sns.set(style='whitegrid',color_codes=True)

#Working with images
import cv2                  
import numpy as np
from tqdm import tqdm
import os                   
from random import shuffle  
from zipfile import ZipFile
from PIL import Image
import shutil

import tensorflow as tf

In [3]:
class CONFIG:
    HEIGHT = 224
    WIDTH = 224

## Organizing Images into seperate directories

## Task 1

In [4]:
#image directories 
TRAIN_DIR = '../dataset/train/images'
TRAIN_LABEL_DIR = '../dataset/train/labels/labels.csv'

VAL_DIR = '../dataset/val/images'
VAL_LABEL_DIR = '../dataset/val/labels/labels.csv'

TEST_DIR = '../dataset/test/images'
TEST_LABEL_DIR = '../dataset/test/labels/labels.csv'

In [5]:
#image labels 
train_labels = pd.read_csv('../dataset/train/labels/labels.csv')
val_labels = pd.read_csv('../dataset/val/labels/labels.csv')
test_labels = pd.read_csv('../dataset/test/labels/labels.csv')
train_labels.head()

,image_id,melanoma,seborrheic_keratosis
0,ISIC_0000000,0.0,0.0
1,ISIC_0000001,0.0,0.0
2,ISIC_0000002,1.0,0.0
3,ISIC_0000003,0.0,0.0
4,ISIC_0000004,1.0,0.0


In [6]:
datasets = {
    'train': {
        'images_dir': TRAIN_DIR,
        'labels_dir': TRAIN_LABEL_DIR
    },
    'val': {
        'images_dir': VAL_DIR,
        'labels_dir': VAL_LABEL_DIR
    },
    'test': {
        'images_dir': TEST_DIR,
        'labels_dir': TEST_LABEL_DIR
    } 
} 

def organize (set, images_dir, labels_dir):
    task1_melanoma_dir = f'../dataset/task1/{set}/melanoma'
    task1_nevus_keratosis_dir = f'../dataset/task1/{set}/nevus_keratosis'
    task2_keratosis_dir = f'../dataset/task2/{set}/keratosis'
    task2_melanoma_nevos_dir = f'../dataset/task2/{set}/melanoma_nevus'
    
    #Make Dirs if don't exist saving_path = "../dataset/task1/train/melanoma/"
    os.makedirs(task1_melanoma_dir, exist_ok = True)
    os.makedirs(task1_nevus_keratosis_dir, exist_ok = True)
    os.makedirs(task2_keratosis_dir, exist_ok = True)
    os.makedirs(task2_melanoma_nevos_dir, exist_ok = True)
    
    #load CSV/ground truth file
    labels = pd.read_csv(labels_dir)
    
    #loop through ground truth and move images based on labels
    for index, row in labels.iterrows():
        image_id = row[0]
        task1_label = row[1]
        task2_label = row[2]
        img_name = f'{image_id}.jpg'
        img_dir = os.path.join(images_dir, img_name)
        if not os.path.exists(images_dir):
            print(f"Image {images_dir} does not exist, skipping...")
            continue
        if task1_label == 1:
            shutil.copy(img_dir, os.path.join(task1_melanoma_dir, img_name))
        else:
            shutil.copy(img_dir, os.path.join(task1_nevus_keratosis_dir, img_name))
        if task2_label == 1:
            shutil.copy(img_dir, os.path.join(task2_keratosis_dir, img_name))
        else: 
            shutil.copy(img_dir, os.path.join(task2_melanoma_nevos_dir, img_name))
    print(f"{set.capitalize()} images organised successfully")

for set, dir in datasets.items():
    organize(set, dir['images_dir'], dir['labels_dir'])


Train images organised successfully
Val images organised successfully
Test images organised successfully


## 2. Preparing TFRecords

In [13]:
def load_data(path, class_names):
    ds = tf.keras.preprocessing.image_dataset_from_directory(
        path,
        labels='inferred', #generate from directory structure
        label_mode = 'binary', #describing the encoding of labels
        class_names = class_names,
        batch_size = 32,
        image_size = (CONFIG.HEIGHT, CONFIG.WIDTH),
        shuffle = True, 
        seed = None,
        validation_split = None,
        subset = None,
        interpolation = 'bilinear', # resizing method
    ) 
    return ds 

In [19]:
#NC 
def _bytes_feature(value):
    """Returns a bytes_list from a string / byte."""
    return tf.train.Feature(bytes_list=tf.train.BytesList(value=[value]))

def _float_feature(value):
    """Returns a float_list from a float / double."""
    return tf.train.Feature(float_list=tf.train.FloatList(value=[value]))

def image_example(image, label):
    # Cast image to uint8
    image = tf.cast(image, tf.uint8)
    image_string = tf.io.encode_jpeg(image).numpy()
    feature = {
        'image': _bytes_feature(image_string),
        'label': _float_feature(label),
    }
    return tf.train.Example(features=tf.train.Features(feature=feature))

def serialize_example(image, label):
    example = image_example(image, label)
    return example.SerializeToString()

def write_tfrecord(dataset, file_pattern, records_per_file):
    # Automatically create the parent directory if it does not exist
    import os
    os.makedirs(os.path.dirname(file_pattern), exist_ok=True)
    
    count = 0
    file_index = 0
    writer = tf.io.TFRecordWriter(file_pattern % file_index)
    
    for images, labels in dataset:
        for image, label in zip(images, labels):
            serialized_example = serialize_example(image, label)
            writer.write(serialized_example)
            count += 1
            
            if count >= records_per_file:
                writer.close()
                count = 0
                file_index += 1
                writer = tf.io.TFRecordWriter(file_pattern % file_index)
    
    writer.close() 

## Task 1

In [20]:
DATA_DIR = '../dataset/task1'
TRAIN_DIR = '../dataset/task1/train'
VAL_DIR = '../dataset/task1/val'
TEST_DIR = '../dataset/task1/test'

In [21]:
class_names = ['nevus_keratosis','melanoma']
train_data = load_data(TRAIN_DIR, class_names)
val_data = load_data(VAL_DIR, class_names)
test_data = load_data(TEST_DIR, class_names)

Found 2000 files belonging to 2 classes.
Found 150 files belonging to 2 classes.
Found 600 files belonging to 2 classes.


In [22]:
#Merge Train and Val data
data = train_data.concatenate(val_data)

In [23]:
# Example usage
train_tf = '../dataset/task1/tf_records/kfold/train%.2i.tfrec'
#val_tf = '../dataset/task1/tf_records/val%.2i.tfrec'
test_tf = '../dataset/task1/tf_records/kfold/test%.2i.tfrec'
records_per_file = 2151 #number of images in one tf file 

write_tfrecord(data, train_tf, records_per_file)
#write_tfrecord(val_data, val_tf, records_per_file)
write_tfrecord(test_data, test_tf, records_per_file)

## Task 2

In [13]:
def load_data(path):
    ds = tf.keras.preprocessing.image_dataset_from_directory(
        path,
        labels='inferred', #generate from directory structure
        label_mode = 'binary', #describing the encoding of labels
        class_names = ['melanoma_nevus','keratosis'],
        batch_size = 32,
        image_size = (CONFIG.HEIGHT, CONFIG.WIDTH),
        shuffle = True, 
        seed = None,
        validation_split = None,
        subset = None,
        interpolation = 'bilinear', # resizing method
    ) 
    return ds 

In [14]:
DATA_DIR = '../dataset/task2'
TRAIN_DIR = '../dataset/task2/train'
VAL_DIR = '../dataset/task2/val'
TEST_DIR = '../dataset/task2/test'
class_names = ['melanoma_nevus','keratosis']
train_data = load_data(TRAIN_DIR, class_names)
val_data = load_data(VAL_DIR, class_names)
test_data = load_data(TEST_DIR, class_names)

Found 2000 files belonging to 2 classes.
Found 150 files belonging to 2 classes.
Found 600 files belonging to 2 classes.


In [9]:
#Merge Train and Val data
data = train_data.concatenate(val_data)

In [16]:
# Example usage
train_tf = '../dataset/task2/tf_records/train%.2i.tfrec'
val_tf = '../dataset/task2/tf_records/val%.2i.tfrec'
test_tf = '../dataset/task2/tf_records/test%.2i.tfrec'
records_per_file = 2151 #number of images in one file 

write_tfrecord(train_data, train_tf, records_per_file)
write_tfrecord(val_data, val_tf, records_per_file)
write_tfrecord(test_data, test_tf, records_per_file)

In [6]:
os.listdir('../dataset/task1/original_data') 

['test', 'train', 'val']